In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution
import seaborn as sns
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
room_type_order = ["cross", "corner", "dual", "single"]
sigma_q_label = "$\\sigma_{q_n}$ Bulk Fit PN"
sigma_p_label = "$\\sigma_{\\Delta C_p}$ Bulk Fit PN"
model_order = [sigma_q_label, sigma_p_label]
model_tick_labels = {
    sigma_q_label: "$\\sigma_{q_n}$",
    sigma_p_label: "$\\sigma_{\\Delta C_p}$",
}


In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

flowStatsMI["q_model-Norm-Adjusted"] = xAdjusted

## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/ashrae_exports/windowASHRAE.csv")
data=flowStatsMI

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plot_df = data[data["slAll"] == False].copy()

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True, layout="constrained")

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    (xAdjusted, "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        alpha=0.7,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


handles, labels = axs[0].get_legend_handles_labels()
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# fig.legend(
#     handles,
#     labels,
#     loc="center left",
#     bbox_to_anchor=(0.90, 0.5),
#     fontsize=12,
#     # title="Room A / Window Type",
#     title_fontsize=13,
#     frameon=False,
# )

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.86, top=0.83, bottom=0.18, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/ashrae_exports/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False], x="ASHRAE", y=y_var, hue="AofA", palette="viridis")

## New Plots

In [ ]:
fig, ax, sigma_q_adjusted, sigma_q_fitted_params = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name=sigma_q_label,
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)
plt.close(fig)

fig, ax, sigma_p_adjusted, sigma_p_fitted_params = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name=sigma_p_label,
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)
plt.close(fig)

flowStatsMI["sigma_q_bulk_fit_pn"] = sigma_q_adjusted
flowStatsMI["sigma_p_bulk_fit_pn"] = sigma_p_adjusted


## ASHRAE PN Bulk-Fit Comparison

In [ ]:
def attach_window_ashrae(frame, ashrae_frame):
    ashrae_lookup = ashrae_frame.set_index(["windowType", "roomA", "C"])["ventilationRate"]
    out = frame.copy()
    out["ASHRAE"] = out.apply(
        lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan)
        if not row["slAll"] else np.nan,
        axis=1,
    )
    return out

def attach_room_ashrae(frame, ashrae_frame):
    ashrae_lookup = ashrae_frame.set_index(["roomType", "AofA", "C"])["ventilationRate"]
    out = frame.copy()
    out["ASHRAE"] = out.apply(
        lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan)
        if not row["slAll"] else np.nan,
        axis=1,
    )
    return out

def compute_error_metrics(frame, pred_col, model_name, *, group_col=None, group_specs=None):
    rows = []
    if group_specs is None:
        if group_col is None:
            raise ValueError("Either group_col or group_specs must be provided.")
        group_specs = []
        for group in frame[group_col].dropna().unique():
            group_specs.append(("group", group, frame[group_col] == group))

    group_specs = [("total", "All", pd.Series(True, index=frame.index))] + list(group_specs)

    for scope, group, mask in group_specs:
        subset = frame.loc[mask, ["ASHRAE", pred_col]].dropna()
        if subset.empty:
            continue

        error = subset[pred_col] - subset["ASHRAE"]
        rmse = np.sqrt(np.mean(error ** 2))
        ashrae_std = np.std(subset["ASHRAE"])
        nrmse = rmse / ashrae_std if ashrae_std > 0 else np.nan
        rows.append(
            {
                "model_name": model_name,
                "scope": scope,
                "group": group,
                "rmse": rmse,
                "bias": error.mean(),
                "nrmse": nrmse,
                "n": len(subset),
            }
        )

    return pd.DataFrame(rows)

def signed_flow_group_specs(frame, sign_col="Sdelp"):
    return [
        ("group", "Flow Entering", frame[sign_col] > 0),
        ("group", "Flow Exiting", frame[sign_col] < 0),
    ]

def _plot_point(ax, y_pos, value, color):
    if pd.isna(value):
        return
    ax.scatter(
        value,
        y_pos,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        s=42,
        zorder=3,
    )

def _plot_connector(ax, y_pos, values):
    clean_values = [value for value in values if pd.notna(value)]
    if len(clean_values) < 2:
        return
    ax.plot(
        clean_values,
        [y_pos] * len(clean_values),
        color="0.35",
        linewidth=1.2,
        alpha=0.25,
        zorder=2,
    )

def add_ashrae_model_overlay(ax, frame, x_col, y_col, model_func, p0, bounds, *, min_points=6, show_legend=False):
    overlay_handles = []
    overlay_labels = []

    for s in [1, -1]:
        if s > 0:
            mask = (frame[x_col] > 0) & frame[x_col].notna() & frame[y_col].notna()
        else:
            mask = (frame[x_col] < 0) & frame[x_col].notna() & frame[y_col].notna()

        regdf = frame.loc[mask, [x_col, y_col]].copy()
        if regdf.empty:
            continue

        regdf_abs = regdf.abs().sort_values(by=x_col)
        if len(regdf_abs) < min_points:
            continue

        try:
            popt = curve_fit(model_func, regdf_abs[x_col], regdf_abs[y_col], p0=p0, bounds=bounds)[0]
        except Exception:
            continue

        x_plot = regdf_abs[x_col].to_numpy() * s
        y_fit = model_func(regdf_abs[x_col].to_numpy(), *popt) * s
        y_upper = model_func(regdf_abs[x_col].to_numpy(), *popt, I_crit=1000) * s
        y_lower = model_func(regdf_abs[x_col].to_numpy(), *popt, I_crit=0.001) * s

        model_line, = ax.plot(x_plot, y_fit, color="k", linewidth=2.2, zorder=4)
        upper_line, = ax.plot(x_plot, y_upper, color="blue", linestyle="--", linewidth=1.3, zorder=4)
        lower_line, = ax.plot(x_plot, y_lower, color="green", linestyle="--", linewidth=1.3, zorder=4)

        if show_legend and s > 0:
            overlay_handles.extend([model_line, upper_line, lower_line])
            overlay_labels.extend(["Model", "Mean Dominated", "Fluctuation Dominated"])

    return overlay_handles, overlay_labels

def compute_ashrae_model_corrected_series(frame, source_col, target_col, model_func, p0, bounds, *, min_points=6):
    corrected = pd.Series(np.nan, index=frame.index, dtype=float)

    for s in [1, -1]:
        if s > 0:
            mask = (frame[source_col] > 0) & frame[source_col].notna() & frame[target_col].notna()
        else:
            mask = (frame[source_col] < 0) & frame[source_col].notna() & frame[target_col].notna()

        regdf = frame.loc[mask, [source_col, target_col]].copy()
        if regdf.empty:
            continue

        regdf_abs = regdf.abs().sort_values(by=source_col)
        if len(regdf_abs) < min_points:
            continue

        try:
            popt = curve_fit(model_func, regdf_abs[source_col], regdf_abs[target_col], p0=p0, bounds=bounds)[0]
        except Exception:
            continue

        corrected.loc[mask] = model_func(frame.loc[mask, source_col].abs().to_numpy(), *popt) * s

    return corrected

def fitted_params_to_df(params_dict, model_name, fit_source):
    rows = []
    for (opening_group, direction), popt in params_dict.items():
        rows.append(
            {
                "fit_source": fit_source,
                "model_name": model_name,
                "opening_group": opening_group,
                "direction": direction,
                "cd": Cd * popt[0],
                "sigma": popt[1] if len(popt) > 1 else np.nan,
            }
        )
    return pd.DataFrame(rows)

def fit_signed_model_params(frame, x_col, y_col, model_func, p0, bounds, *, min_points=6, opening_group="Window"):
    rows = []
    params = {}

    for s, direction in [(1, "Flow Entering"), (-1, "Flow Exiting")]:
        if s > 0:
            mask = (frame[x_col] > 0) & frame[x_col].notna() & frame[y_col].notna()
        else:
            mask = (frame[x_col] < 0) & frame[x_col].notna() & frame[y_col].notna()

        regdf = frame.loc[mask, [x_col, y_col]].copy()
        if regdf.empty:
            continue

        regdf_abs = regdf.abs().sort_values(by=x_col)
        if len(regdf_abs) < min_points:
            continue

        try:
            popt = curve_fit(model_func, regdf_abs[x_col], regdf_abs[y_col], p0=p0, bounds=bounds)[0]
        except Exception:
            continue

        params[(opening_group, direction)] = popt
        rows.append(
            {
                "opening_group": opening_group,
                "direction": direction,
                "cd": Cd * popt[0],
                "sigma": popt[1] if len(popt) > 1 else np.nan,
                "n": len(regdf_abs),
                "x_label": x_col,
                "y_label": y_col,
            }
        )

    return pd.DataFrame(rows), params

def plot_error_summary(metrics_df, title, group_palette, model_order, model_tick_labels):
    metric_rows = [("rmse", "RMSE"), ("bias", "Bias")]
    fig, axs = plt.subplots(2, 1, figsize=(8.2, 7.8), dpi=160, sharey=True)
    group_data = metrics_df[metrics_df["scope"] == "group"].copy()
    group_order = [group for group in group_palette if group in group_data["group"].unique()]

    for row_idx, (metric_col, metric_label) in enumerate(metric_rows):
        ax = axs[row_idx]
        for model_idx, model_name in enumerate(model_order):
            point_values = []
            for group in group_order:
                row = group_data[
                    (group_data["model_name"] == model_name)
                    & (group_data["group"] == group)
                ]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                point_values.append(value)
                _plot_point(ax, model_idx, value, group_palette[group])
            _plot_connector(ax, model_idx, point_values)

        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        ax.set_yticks(range(len(model_order)))
        ax.set_yticklabels([model_tick_labels[name] for name in model_order], fontsize=16)
        ax.tick_params(axis="x", labelsize=14)
        ax.tick_params(axis="y", labelsize=16)
        ax.set_ylim(-0.5, len(model_order) - 0.5)
        ax.set_xlabel(metric_label, fontsize=16)
        if row_idx == 0:
            ax.set_title(title, fontsize=18)

    handles = [
        Line2D([0], [0], color=color, marker="o", linestyle="None", markersize=9, label=group)
        for group, color in group_palette.items()
        if group in group_order
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=max(1, len(handles)),
        fontsize=12,
        frameon=False,
    )
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    return fig, axs

def plot_combined_error_summary(window_metrics_df, room_metrics_df, window_palette, room_palette, model_order, model_tick_labels):
    metric_rows = [("rmse", "RMSE"), ("bias", "Bias")]
    fig, axs = plt.subplots(2, 2, figsize=(13.5, 9.0), dpi=160, sharey=True)
    panel_specs = [
        (window_metrics_df, window_palette, "Windows"),
        (room_metrics_df, room_palette, "Rooms"),
    ]

    for col_idx, (metrics_df, group_palette, title) in enumerate(panel_specs):
        group_data = metrics_df[metrics_df["scope"] == "group"].copy()
        group_order = [group for group in group_palette if group in group_data["group"].unique()]

        for row_idx, (metric_col, metric_label) in enumerate(metric_rows):
            ax = axs[row_idx, col_idx]
            for model_idx, model_name in enumerate(model_order):
                point_values = []
                for group in group_order:
                    row = group_data[
                        (group_data["model_name"] == model_name)
                        & (group_data["group"] == group)
                    ]
                    if row.empty:
                        continue
                    value = row.iloc[0][metric_col]
                    point_values.append(value)
                    _plot_point(ax, model_idx, value, group_palette[group])
                _plot_connector(ax, model_idx, point_values)

            ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
            ax.grid(axis="x", alpha=0.25)
            ax.grid(axis="y", alpha=0.35, linestyle="--")
            ax.set_yticks(range(len(model_order)))
            ax.set_yticklabels([model_tick_labels[name] for name in model_order], fontsize=15)
            ax.tick_params(axis="x", labelsize=14)
            ax.tick_params(axis="y", labelsize=15)
            ax.set_ylim(-0.5, len(model_order) - 0.5)
            ax.set_xlabel(metric_label, fontsize=15)
            if row_idx == 0:
                ax.set_title(title, fontsize=18)

    window_groups = window_metrics_df[window_metrics_df["scope"] == "group"]["group"].unique()
    room_groups = room_metrics_df[room_metrics_df["scope"] == "group"]["group"].unique()
    window_handles = [
        Line2D([0], [0], color=color, marker="o", linestyle="None", markersize=8, label=group)
        for group, color in window_palette.items()
        if group in window_groups
    ]
    room_handles = [
        Line2D([0], [0], color=color, marker="o", linestyle="None", markersize=8, label=group)
        for group, color in room_palette.items()
        if group in room_groups
    ]
    if window_handles:
        fig.legend(window_handles, [h.get_label() for h in window_handles], loc="lower left", bbox_to_anchor=(0.06, 0.01), ncol=max(1, len(window_handles)), fontsize=10, frameon=False)
    if room_handles:
        fig.legend(room_handles, [h.get_label() for h in room_handles], loc="lower right", bbox_to_anchor=(0.94, 0.01), ncol=max(1, len(room_handles)), fontsize=10, frameon=False)
    fig.tight_layout(rect=[0.02, 0.08, 0.98, 1])
    return fig, axs

window_plot_df = attach_window_ashrae(flowStatsMI, windowASHRAE)
window_plot_df = window_plot_df[window_plot_df["slAll"] == False].copy()
window_plot_df["sigma_q_ashrae_corrected"] = compute_ashrae_model_corrected_series(
    window_plot_df,
    "sigma_q_bulk_fit_pn",
    "ASHRAE",
    pyafn.ventilationReDecomp_q,
    [1.0, 0.1],
    ([0.1, 0], [np.inf, np.inf]),
)
window_plot_df["sigma_p_ashrae_corrected"] = compute_ashrae_model_corrected_series(
    window_plot_df,
    "sigma_p_bulk_fit_pn",
    "ASHRAE",
    pyafn.ventilationReDecomp_p,
    [1.0, 0.01],
    ([0.1, 0], [np.inf, np.inf]),
)

room_plot_df = aggregate_window_series_to_room(
    roomVentilationMI,
    flowStatsMI["sigma_q_bulk_fit_pn"],
    out_col="sigma_q_bulk_fit_pn",
    mode="abs_half_sum",
)
room_plot_df = aggregate_window_series_to_room(
    room_plot_df,
    flowStatsMI["sigma_p_bulk_fit_pn"],
    out_col="sigma_p_bulk_fit_pn",
    mode="abs_half_sum",
)
room_plot_df = aggregate_window_series_to_room(
    room_plot_df,
    window_plot_df["sigma_q_ashrae_corrected"],
    out_col="sigma_q_ashrae_corrected_room",
    mode="abs_half_sum",
)
room_plot_df = aggregate_window_series_to_room(
    room_plot_df,
    window_plot_df["sigma_p_ashrae_corrected"],
    out_col="sigma_p_ashrae_corrected_room",
    mode="abs_half_sum",
)
room_plot_df = attach_room_ashrae(room_plot_df, roomASHRAE)
room_plot_df = room_plot_df[room_plot_df["slAll"] == False].copy()

window_flow_groups = signed_flow_group_specs(window_plot_df)
window_error_palette = {
    "Flow Entering": "red",
    "Flow Exiting": "blue",
}

window_error_metrics_df = pd.concat(
    [
        compute_error_metrics(window_plot_df, "sigma_q_bulk_fit_pn", sigma_q_label, group_specs=window_flow_groups),
        compute_error_metrics(window_plot_df, "sigma_p_bulk_fit_pn", sigma_p_label, group_specs=window_flow_groups),
    ],
    ignore_index=True,
)
sigma_q_corrected_label = "$\\sigma_{q_n}$ ASHRAE-Corrected"
sigma_p_corrected_label = "$\\sigma_{\\Delta C_p}$ ASHRAE-Corrected"
corrected_model_order = [sigma_q_corrected_label, sigma_p_corrected_label]
corrected_model_tick_labels = {
    sigma_q_corrected_label: "$\\sigma_{q_n}$ corrected",
    sigma_p_corrected_label: "$\\sigma_{\\Delta C_p}$ corrected",
}
combined_model_order = [sigma_q_label, sigma_p_label, sigma_q_corrected_label, sigma_p_corrected_label]
combined_model_tick_labels = {
    sigma_q_label: "$\\sigma_{q_n}$",
    sigma_p_label: "$\\sigma_{\\Delta C_p}$",
    sigma_q_corrected_label: "$\\sigma_{q_n}$ corrected",
    sigma_p_corrected_label: "$\\sigma_{\\Delta C_p}$ corrected",
}
window_corrected_error_metrics_df = pd.concat(
    [
        compute_error_metrics(window_plot_df, "sigma_q_ashrae_corrected", sigma_q_corrected_label, group_specs=window_flow_groups),
        compute_error_metrics(window_plot_df, "sigma_p_ashrae_corrected", sigma_p_corrected_label, group_specs=window_flow_groups),
    ],
    ignore_index=True,
)

room_error_metrics_df = pd.concat(
    [
        compute_error_metrics(room_plot_df, "sigma_q_bulk_fit_pn", sigma_q_label, group_col="roomType"),
        compute_error_metrics(room_plot_df, "sigma_p_bulk_fit_pn", sigma_p_label, group_col="roomType"),
    ],
    ignore_index=True,
)
room_corrected_error_metrics_df = pd.concat(
    [
        compute_error_metrics(room_plot_df, "sigma_q_ashrae_corrected_room", sigma_q_corrected_label, group_col="roomType"),
        compute_error_metrics(room_plot_df, "sigma_p_ashrae_corrected_room", sigma_p_corrected_label, group_col="roomType"),
    ],
    ignore_index=True,
)
window_combined_error_metrics_df = pd.concat(
    [window_error_metrics_df, window_corrected_error_metrics_df],
    ignore_index=True,
)
room_combined_error_metrics_df = pd.concat(
    [room_error_metrics_df, room_corrected_error_metrics_df],
    ignore_index=True,
)
pn_fit_params_df = pd.concat(
    [
        fitted_params_to_df(sigma_q_fitted_params, sigma_q_label, "PN Bulk Fit"),
        fitted_params_to_df(sigma_p_fitted_params, sigma_p_label, "PN Bulk Fit"),
    ],
    ignore_index=True,
)

sigma_q_ashrae_overlay_fit_params_df, sigma_q_ashrae_overlay_fit_params = fit_signed_model_params(
    window_plot_df,
    "ASHRAE",
    "sigma_q_bulk_fit_pn",
    pyafn.ventilationReDecomp_q,
    [1.0, 0.1],
    ([0.1, 0], [np.inf, np.inf]),
)
sigma_q_ashrae_overlay_fit_params_df["fit_source"] = "ASHRAE Overlay Fit"
sigma_q_ashrae_overlay_fit_params_df["model_name"] = sigma_q_label

sigma_p_ashrae_overlay_fit_params_df, sigma_p_ashrae_overlay_fit_params = fit_signed_model_params(
    window_plot_df,
    "ASHRAE",
    "sigma_p_bulk_fit_pn",
    pyafn.ventilationReDecomp_p,
    [1.0, 0.01],
    ([0.1, 0], [np.inf, np.inf]),
)
sigma_p_ashrae_overlay_fit_params_df["fit_source"] = "ASHRAE Overlay Fit"
sigma_p_ashrae_overlay_fit_params_df["model_name"] = sigma_p_label

ashrae_overlay_fit_params_df = pd.concat(
    [sigma_q_ashrae_overlay_fit_params_df, sigma_p_ashrae_overlay_fit_params_df],
    ignore_index=True,
)

sigma_q_ashrae_correction_fit_params_df, sigma_q_ashrae_correction_fit_params = fit_signed_model_params(
    window_plot_df,
    "sigma_q_bulk_fit_pn",
    "ASHRAE",
    pyafn.ventilationReDecomp_q,
    [1.0, 0.1],
    ([0.1, 0], [np.inf, np.inf]),
)
sigma_q_ashrae_correction_fit_params_df["fit_source"] = "ASHRAE Correction Fit"
sigma_q_ashrae_correction_fit_params_df["model_name"] = sigma_q_corrected_label

sigma_p_ashrae_correction_fit_params_df, sigma_p_ashrae_correction_fit_params = fit_signed_model_params(
    window_plot_df,
    "sigma_p_bulk_fit_pn",
    "ASHRAE",
    pyafn.ventilationReDecomp_p,
    [1.0, 0.01],
    ([0.1, 0], [np.inf, np.inf]),
)
sigma_p_ashrae_correction_fit_params_df["fit_source"] = "ASHRAE Correction Fit"
sigma_p_ashrae_correction_fit_params_df["model_name"] = sigma_p_corrected_label

ashrae_correction_fit_params_df = pd.concat(
    [sigma_q_ashrae_correction_fit_params_df, sigma_p_ashrae_correction_fit_params_df],
    ignore_index=True,
)

fit_params_df = pd.concat(
    [pn_fit_params_df, ashrae_overlay_fit_params_df, ashrae_correction_fit_params_df],
    ignore_index=True,
)
ashrae_fit_params_export_df = pd.concat(
    [ashrae_overlay_fit_params_df, ashrae_correction_fit_params_df],
    ignore_index=True,
)[["fit_source", "model_name", "opening_group", "direction", "cd", "sigma", "n", "x_label", "y_label"]]
ashrae_error_metrics_export_df = pd.concat(
    [
        window_error_metrics_df.assign(dataset="windows", evaluation="bulk_fit"),
        room_error_metrics_df.assign(dataset="rooms", evaluation="bulk_fit"),
        window_corrected_error_metrics_df.assign(dataset="windows", evaluation="ashrae_corrected"),
        room_corrected_error_metrics_df.assign(dataset="rooms", evaluation="ashrae_corrected"),
    ],
    ignore_index=True,
)[["dataset", "evaluation", "model_name", "scope", "group", "rmse", "bias", "nrmse", "n"]]

window_match_summary = pd.DataFrame(
    {
        "dataset": ["windows"],
        "rows": [len(window_plot_df)],
        "ashrae_matches": [window_plot_df["ASHRAE"].notna().sum()],
    }
)
room_match_summary = pd.DataFrame(
    {
        "dataset": ["rooms"],
        "rows": [len(room_plot_df)],
        "ashrae_matches": [room_plot_df["ASHRAE"].notna().sum()],
    }
)


In [ ]:
window_type_order = list(window_plot_df["windowType"].dropna().unique())
window_palette = dict(zip(window_type_order, sns.color_palette("tab10", n_colors=len(window_type_order))))
room_palette = {
    room_type: color
    for room_type, color in zip(room_type_order, sns.color_palette("colorblind", n_colors=len(room_type_order)))
    if room_type in room_plot_df["roomType"].dropna().unique()
}

scatter_specs = [
    (sigma_q_label, "sigma_q_bulk_fit_pn"),
    (sigma_p_label, "sigma_p_bulk_fit_pn"),
]

scatter_limit_values = [
    window_plot_df["ASHRAE"],
    room_plot_df["ASHRAE"],
    window_plot_df["sigma_q_bulk_fit_pn"],
    window_plot_df["sigma_p_bulk_fit_pn"],
    room_plot_df["sigma_q_bulk_fit_pn"],
    room_plot_df["sigma_p_bulk_fit_pn"],
]
scatter_limit_values = np.concatenate([series.dropna().to_numpy() for series in scatter_limit_values if not series.dropna().empty])
scatter_min = np.nanmin(scatter_limit_values)
scatter_max = np.nanmax(scatter_limit_values)
room_scatter_max = np.nanmax(np.concatenate([
    room_plot_df["ASHRAE"].dropna().to_numpy(),
    room_plot_df["sigma_q_bulk_fit_pn"].dropna().to_numpy(),
    room_plot_df["sigma_p_bulk_fit_pn"].dropna().to_numpy(),
    room_plot_df["sigma_q_ashrae_corrected_room"].dropna().to_numpy(),
    room_plot_df["sigma_p_ashrae_corrected_room"].dropna().to_numpy(),
]))

fig_fit, axs_fit = plt.subplots(2, 3, figsize=(16.0, 9.5), dpi=160, sharex=False, sharey=False)
overlay_handles = []
overlay_labels = []

for row_idx, (title, pred_col) in enumerate(scatter_specs):
    sns.scatterplot(
        data=window_plot_df,
        x="ASHRAE",
        y=pred_col,
        hue="roomType",
        palette=room_palette,
        hue_order=[room for room in room_type_order if room in room_palette],
        alpha=0.75,
        s=26,
        ax=axs_fit[row_idx, 0],
        legend=row_idx == 0,
    )
    handles, labels = add_ashrae_model_overlay(
        axs_fit[row_idx, 0],
        window_plot_df,
        "ASHRAE",
        pred_col,
        pyafn.ventilationReDecomp_q if pred_col == "sigma_q_bulk_fit_pn" else pyafn.ventilationReDecomp_p,
        [1.0, 0.1] if pred_col == "sigma_q_bulk_fit_pn" else [1.0, 0.01],
        ([0.1, 0], [np.inf, np.inf]),
        show_legend=(row_idx == 0),
    )
    if handles:
        overlay_handles = handles
        overlay_labels = labels
    axs_fit[row_idx, 0].plot([scatter_min, scatter_max], [scatter_min, scatter_max], "r--", linewidth=1.2, alpha=0.7)
    axs_fit[row_idx, 0].set_ylabel("Prediction", fontsize=16)
    axs_fit[row_idx, 0].grid(True, alpha=0.3)
    axs_fit[row_idx, 0].tick_params(labelsize=13)
    axs_fit[row_idx, 0].set_xlim(scatter_min, scatter_max)
    axs_fit[row_idx, 0].set_ylim(scatter_min, scatter_max)

    sns.scatterplot(
        data=room_plot_df,
        x="ASHRAE",
        y=pred_col,
        hue="roomType",
        palette=room_palette,
        hue_order=[room for room in room_type_order if room in room_palette],
        alpha=0.75,
        s=26,
        ax=axs_fit[row_idx, 1],
        legend=row_idx == 0,
    )
    axs_fit[row_idx, 1].plot([0, room_scatter_max], [0, room_scatter_max], "r--", linewidth=1.2, alpha=0.7)
    axs_fit[row_idx, 1].grid(True, alpha=0.3)
    axs_fit[row_idx, 1].tick_params(labelsize=13)
    axs_fit[row_idx, 1].set_xlim(0, room_scatter_max)
    axs_fit[row_idx, 1].set_ylim(0, room_scatter_max)

    corrected_room_col = "sigma_q_ashrae_corrected_room" if pred_col == "sigma_q_bulk_fit_pn" else "sigma_p_ashrae_corrected_room"
    sns.scatterplot(
        data=room_plot_df,
        x="ASHRAE",
        y=corrected_room_col,
        hue="roomType",
        palette=room_palette,
        hue_order=[room for room in room_type_order if room in room_palette],
        alpha=0.75,
        s=26,
        ax=axs_fit[row_idx, 2],
        legend=False,
    )
    axs_fit[row_idx, 2].plot([0, room_scatter_max], [0, room_scatter_max], "r--", linewidth=1.2, alpha=0.7)
    axs_fit[row_idx, 2].grid(True, alpha=0.3)
    axs_fit[row_idx, 2].tick_params(labelsize=13)
    axs_fit[row_idx, 2].set_xlim(0, room_scatter_max)
    axs_fit[row_idx, 2].set_ylim(0, room_scatter_max)

for ax in [axs_fit[1, 0], axs_fit[1, 1], axs_fit[1, 2]]:
    ax.set_xlabel("ASHRAE Ventilation Rate", fontsize=16)

for ax in [axs_fit[0, 1], axs_fit[1, 1], axs_fit[0, 2], axs_fit[1, 2]]:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

axs_fit[0, 0].set_title("Windows", fontsize=18)
axs_fit[0, 1].set_title("Rooms (Skylights Closed)", fontsize=18)
axs_fit[0, 2].set_title("ASHRAE Model-Corrected Rooms", fontsize=18)
axs_fit[0, 0].text(-0.26, 0.5, "$\\sigma_{q_n}$", rotation=90, transform=axs_fit[0, 0].transAxes, va="center", ha="center", fontsize=18)
axs_fit[1, 0].text(-0.26, 0.5, "$\\sigma_{\\Delta C_p}$", rotation=90, transform=axs_fit[1, 0].transAxes, va="center", ha="center", fontsize=18)
if overlay_handles:
    fig_fit.legend(overlay_handles, overlay_labels, loc="lower center", bbox_to_anchor=(0.5, 0.01), ncol=3, fontsize=12, frameon=False)
fig_fit.suptitle("ASHRAE PN Bulk-Fit Model Comparison", fontsize=20)
fig_fit.tight_layout(rect=[0.04, 0.05, 1, 0.96])


In [ ]:
import os

ashrae_export_dir = os.path.join(home_dir, "ashrae_exports")
os.makedirs(ashrae_export_dir, exist_ok=True)

old_ashrae_export_files = [
    "window_error_metrics.csv",
    "room_error_metrics.csv",
    "window_corrected_error_metrics.csv",
    "room_corrected_error_metrics.csv",
    "window_combined_error_metrics.csv",
    "room_combined_error_metrics.csv",
    "pn_fit_params.csv",
    "ashrae_overlay_fit_params.csv",
    "ashrae_correction_fit_params.csv",
    "fit_params.csv",
]
for filename in old_ashrae_export_files:
    filepath = os.path.join(ashrae_export_dir, filename)
    if os.path.exists(filepath):
        os.remove(filepath)

ashrae_export_tables = {
    "ashrae_error_metrics.csv": ashrae_error_metrics_export_df,
    "ashrae_fit_params.csv": ashrae_fit_params_export_df,
}

for filename, df in ashrae_export_tables.items():
    df.to_csv(os.path.join(ashrae_export_dir, filename), index=False)

ashrae_export_manifest_df = pd.DataFrame(
    {
        "table_name": list(ashrae_export_tables.keys()),
        "rows": [len(df) for df in ashrae_export_tables.values()],
        "path": [os.path.join(ashrae_export_dir, filename) for filename in ashrae_export_tables.keys()],
    }
)

display(window_match_summary)
display(room_match_summary)
display(window_error_metrics_df)
display(room_error_metrics_df)
display(window_corrected_error_metrics_df)
display(room_corrected_error_metrics_df)
display(ashrae_fit_params_export_df)
display(ashrae_export_manifest_df)


In [ ]:
fig_error_combined, axs_error_combined = plot_combined_error_summary(
    window_combined_error_metrics_df,
    room_combined_error_metrics_df,
    window_error_palette,
    room_palette,
    combined_model_order,
    combined_model_tick_labels,
)
